# Voltage Loss Detection with GridFM: deriving and tuning the VLD loss

**gridfm-app-distribution — hands-on Colab tutorial**

A distribution network loses voltage at every bus that no longer has a path to a
source after a contingency. This tutorial teaches the loss function that lets a
graph transformer learn that physical constraint, and then uses it end to end:

1. **Generate data** — download a sample pandapower distribution network
   (`case33bw`) and run N-1/N-2 contingencies with `contingency_data_gen`.
2. **Derive the VLD loss** — reachability labels, class balance, the two BCE
   terms, the conflict rule, the voltage terms, the gradients and the derived
   decision threshold, each shown numerically on one generated `case33bw`
   contingency scenario.
3. **Tune it** — see what each weight changes in the gradients.
4. **Train a dummy model** with the real CLI and read its loss curves.
5. **Run inference with a pre-trained model** and read the evaluation report.

Everything is computed when you run the notebook; nothing below is a claimed result.

## 0. Setup

1. Open in [Google Colab](https://colab.research.google.com/) and select **Runtime → Change runtime type → T4 GPU** (CPU works too, more slowly).
2. Run the cells in order. The install cell takes a few minutes the first time; if Colab asks you to restart the runtime afterwards, do so and re-run from the top (the install is skipped on the second pass).

The application is installed straight from GitHub, so the notebook always runs
against the pushed code.

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
WORK = Path("/content/vld_tutorial") if IN_COLAB else Path.cwd() / "vld_tutorial_work"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

REPO_URL = "https://github.com/gridfm/gridfm-app-distribution"
REPO_DIR = WORK / "gridfm-app-distribution"
TUTORIAL_DIR = REPO_DIR / "examples" / "tutorial_vld"
# Make the package importable in this kernel and in the CLI subprocesses,
# without needing a runtime restart after the editable install.
sys.path.insert(0, str(REPO_DIR))
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
DATA_DIR, LOG_DIR, PRED_DIR = WORK / "data", WORK / "mlruns", WORK / "predictions"
for d in (DATA_DIR, LOG_DIR, PRED_DIR):
    d.mkdir(exist_ok=True)
print("working directory:", WORK)

In [ ]:
# Install the application (once per runtime) --------------------------------
# Colab now ships Python 3.13 while the package declares requires-python <3.13,
# so the version check is skipped; pip's output is shown if anything fails.
marker = WORK / ".installed"
if not marker.exists():
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
        )
    # The pre-trained checkpoint (*.pt) is stored with Git LFS. Without git-lfs
    # the clone holds small pointer files instead of the weights.
    subprocess.run(
        "command -v git-lfs || (apt-get -qq update && apt-get -qq install -y git-lfs)",
        shell=True,
        check=True,
    )
    subprocess.run(["git", "lfs", "install", "--local"], cwd=REPO_DIR, check=True)
    subprocess.run(
        ["git", "lfs", "pull", "--include", "examples/tutorial_vld/pretrained/*"],
        cwd=REPO_DIR,
        check=True,
    )
    pip = [sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python"]
    r = subprocess.run(pip + ["-e", str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-4000:], r.stderr[-4000:])
        raise RuntimeError("pip install failed; see pip's output above")
    # torch-scatter must match the torch build that pip just resolved
    import torch

    tv = torch.__version__.split("+")[0]
    cu = (
        "cpu"
        if torch.version.cuda is None
        else "cu" + torch.version.cuda.replace(".", "")
    )
    subprocess.run(
        pip + ["torch-scatter", "-f", f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"],
        check=True,
    )
    marker.write_text(tv)
    print(
        "Installed. If Colab prompts to restart the runtime, do it and re-run from the top."
    )
else:
    print("Already installed.")

In [ ]:
sys.path.insert(0, str(TUTORIAL_DIR))
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from tutorial_vld_helpers import (
    run,
    python_module,
    latest_run_artifacts,
    plot_history,
    read_yaml,
    write_yaml,
    count_scenarios,
    adapt_pretrained,
    stationary_point,
    pick_scenario,
    scenario_table,
    scenario_roles,
    plot_reachability,
    plot_scenario,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE)

# 1. What is Voltage Loss Detection?

After a bus or branch outage, a bus is **energised** only if (a) it is itself
available and (b) there is a path of available branches through available buses
from an energisation source (the slack / external grid) to it. Everything else
is **de-energised** and its voltage is zero.

The model receives the pre-contingency topology plus the contingency flags and
must predict, per bus, a **status logit** $z_i$ (with $\hat p_i = \sigma(z_i)$
the probability of being energised) and the **voltage magnitude** $\hat V_i$.

Two labels exist for every bus:

* $y_i^{\text{reach}}$ — derived inside the loss by a breadth-first search over
  the available graph (eqs. 1–5 of the paper);
* $y_i^{\text{target}}$ — the power-flow solver's post-contingency flag in the dataset.

They agree almost everywhere; where they do not, the loss has a rule, which we
will get to.

# 2. Generate contingency data

We take a sample distribution network from
[pandapower's built-in networks](https://pandapower.readthedocs.io/en/v2.0.0/networks.html):
**`case33bw`**, the 33-bus radial feeder of Baran & Wu. `contingency_data_gen`
(a second application in this repository) builds it, enumerates N-1 and N-2 bus
and branch outages, solves each post-contingency power flow and writes the
results as gridfm-datakit parquet under `data/<name>/raw/`, the layout the model
reads.

The job is described in `configs/tutorial_datagen.yaml`:

In [ ]:
DATAGEN_CFG = TUTORIAL_DIR / "configs" / "tutorial_datagen.yaml"
print(DATAGEN_CFG.read_text())
run(["contingency_data_gen", "--config", DATAGEN_CFG, "--dry-run"], cwd=WORK)

In [ ]:
NETWORK = "tutorial_case33bw"
if not (DATA_DIR / NETWORK / "raw" / "bus_data.parquet").exists():
    run(
        ["contingency_data_gen", "--config", DATAGEN_CFG],
        cwd=WORK,
        env={"SLURM_CPUS_PER_TASK": "2"},
    )
N_SCEN = count_scenarios(DATA_DIR, NETWORK)
print(f"{N_SCEN} scenarios generated under {DATA_DIR / NETWORK / 'raw'}")

## 2.1 Look at the data

Three tables per scenario. The columns the VLD loss relies on are the bus
`BUS_BASE_STATUS` / `BUS_CONTINGENCY` flags, the branch `BRANCH_BASE_STATUS` /
`BRANCH_CONTINGENCY` flags (inputs to the BFS), the `REF` indicator (the BFS
seed) and the `BUS_STATUS` target. `case33bw` has only an external grid, so there
is no generator table: the model treats the network as generator-free.

In [ ]:
raw = DATA_DIR / NETWORK / "raw"
bus = pd.read_parquet(raw / "bus_data.parquet")
branch = pd.read_parquet(raw / "branch_data.parquet")
print("bus columns:", list(bus.columns))
display(bus.head(8))
per_scen = bus.groupby("scenario").BUS_STATUS.agg(["mean", "size"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(1 - per_scen["mean"], bins=20, color="#1f5fa8")
axes[0].set(xlabel="de-energised fraction per scenario", ylabel="scenarios")
axes[1].hist(bus.loc[bus.BUS_STATUS == 1, "Vm"], bins=30, color="#d9731a")
axes[1].set(xlabel="Vm of energised buses (p.u.)", ylabel="bus records")
plt.tight_layout()
plt.show()
print(
    f"Overall de-energised fraction: {1 - bus.BUS_STATUS.mean():.3f}  "
    f"->  class ratio n_on/n_off = {bus.BUS_STATUS.mean() / (1 - bus.BUS_STATUS.mean()):.2f}"
)

That last number matters: it is $w_{\text{off}} = n_{\text{on}}/n_{\text{off}}$,
the class weight the loss will apply to de-energised buses, and it fixes the
decision threshold the model will use. Keep it in mind.

# 3. The VLD loss, derived

$$\mathcal{L}_{\text{VLD}} = \lambda_{\text{reach}}\,\mathcal{L}_{\text{reach}} + \lambda_{\text{anc}}\,\mathcal{L}_{\text{anc}} + \lambda_{\text{off}}\,\mathcal{L}_{\text{off}} + \lambda_{\text{on}}\,\mathcal{L}_{\text{on}}$$

| term | supervises | target | per-bus weight |
|---|---|---|---|
| $\mathcal{L}_{\text{reach}}$ | $z_i$ | $y_i^{\text{reach}}$ (BFS) | $\omega_i$: $w_{\text{off}}$ on de-energised buses, 1 otherwise; **0 on conflict buses** |
| $\mathcal{L}_{\text{anc}}$ | $z_i$ | $y_i^{\text{target}}$ (solver) | $\nu_i$: $w_{\text{off}}$ on label-0 buses, 1 otherwise; **$w_{\text{off}}$ on conflict buses** |
| $\mathcal{L}_{\text{off}}$ | $\hat V_i \to 0$ | buses with $y_i^{\text{reach}}=0$ | $\hat V_i^2 + \lambda_1 \lvert \hat V_i \rvert$ |
| $\mathcal{L}_{\text{on}}$ | $\hat V_i \to V_i$ | buses with $y_i^{\text{reach}}=1$ | $\eta_i = \mathrm{sg}(\hat p_i)$ (stop-gradient) |

Each term is a weighted mean, so the $\lambda$ values compare like with like.
We build the loss up on **one contingency scenario of the `case33bw` feeder
generated in section 2**, using the **actual** `VLDTopologyLoss` class, not a
re-implementation.

## 3.1 Reachability and class balance

The loss starts from the scenario's flags alone. A breadth-first search seeded at
the REF bus (the external grid) walks only through **available** branches (in
service in the base network and not hit) and **available** buses. Every bus it
reaches gets $y^{\text{reach}}=1$. Every other bus gets 0: either it was hit
directly, or it is healthy but **islanded** downstream of the outage. On a radial
feeder like `case33bw`, one branch outage islands the whole subtree below it.

The cell below picks a scenario that has both kinds of de-energised bus, draws it
and lists the labels. The class weight is $w_{\text{off}} = n_{\text{on}}/n_{\text{off}}$,
counted over this scenario and clamped at `max_pos_weight`. During training it is
counted per batch and smoothed with momentum, so it ends up close to the
dataset-wide ratio from section 2.

In [ ]:
SID = pick_scenario(bus, branch)
tbl = scenario_table(bus, branch, SID)
ROLE = scenario_roles(tbl)  # row positions: ref / on / hit / islanded
plot_reachability(bus, branch, SID)
plt.show()

hit = tbl.bus[tbl.available == 0].tolist()
isl = tbl.bus[(tbl.available == 1) & (tbl.y_reach == 0)].tolist()
print(f"scenario {SID}: buses hit {hit}; healthy but islanded {isl}")
display(
    tbl.loc[tbl.y_reach == 0, ["bus", "available", "y_reach", "y_target", "conflict"]]
)
n_on, n_off = int(tbl.y_reach.sum()), int((tbl.y_reach == 0).sum())
print(
    f"n_on = {n_on}, n_off = {n_off}  ->  w_off = {tbl.attrs['w_off']:.3f}"
    f"{'  (clamped)' if n_off and n_on / n_off > tbl.attrs['w_off'] + 1e-6 else ''}    "
    f"derived threshold theta = 1/(1+w_off) = {tbl.attrs['threshold']:.3f}"
)
print(f"BFS vs solver label conflicts in this scenario: {int(tbl.conflict.sum())}")

## 3.2 The two BCE terms and their gradient

For one bus the gradient of the two BCE terms with respect to the logit is
(paper eq. 19)

$$\frac{\partial \mathcal{L}}{\partial z_i} = a_i\,(\hat p_i - y_i^{\text{reach}}) + b_i\,(\hat p_i - y_i^{\text{target}}),\qquad a_i = \lambda_{\text{reach}}\frac{\omega_i}{\sum_j\omega_j},\quad b_i = \lambda_{\text{anc}}\frac{\nu_i}{\sum_j\nu_j}.$$

Start every bus at $\hat V = 1$ and $z = 2$ ($\hat p = 0.88$, "everything is
on"). A **positive** gradient means gradient descent will push the value down.

In [ ]:
tbl = scenario_table(bus, branch, SID)
# The two de-energised groups first, then the energised buses
show = tbl.sort_values(["y_reach", "available"]).reset_index(drop=True)
display(show)
terms = tbl.attrs["terms"]
display(
    pd.Series(
        {
            k: round(v, 4)
            for k, v in terms.items()
            if k.startswith("VLD") and "BCE" in k or k == "loss"
        }
    )
)

Read the `dL/dz` column: the energised buses ($\hat p < 1$) get a small
negative gradient and move up; the de-energised buses, hit and islanded alike,
get a positive gradient that is $w_{\text{off}}$ times larger in magnitude, so
they move down faster. That asymmetry is the class balance at work.

## 3.3 Where does a bus end up? The stationary point and the threshold

Setting the gradient to zero gives the probability a bus settles at (eq. 20):

$$\hat p_i^\star = \frac{a_i\,y_i^{\text{reach}} + b_i\,y_i^{\text{target}}}{a_i + b_i}.$$

On agreeing buses that is just the label. But the class weight has a side
effect on **uncertain** buses: if the true posterior of a bus is $q = P(\text{on}\mid x)$,
a BCE weighted by $w$ on the 0-class is minimised at

$$\hat p = \frac{q}{q + w\,(1-q)},$$

not at $q$. A fixed threshold of 0.5 would therefore call a bus energised only
when $q \ge w/(1+w)$, i.e. it over-predicts de-energisation. The loss undoes
this by thresholding at

$$\theta = \frac{1}{1 + w_{\text{off}}},$$

which maps the shifted optimum back to $q = 0.5$. **The threshold is not a
tuning parameter**: it is derived from $w_{\text{off}}$, reported by the loss
as `VLD Decision Threshold`, and used by the task at test and predict time.

In [ ]:
w = tbl.attrs["w_off"]
q = np.linspace(0.01, 0.99, 99)
p_hat = q / (q + w * (1 - q))
fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.plot(q, p_hat, color="#1f5fa8", label=rf"$\hat p = q/(q + w(1-q))$, $w$ = {w:.2f}")
ax.plot(q, q, "--", color="#9a9a9a", label="unweighted ($\\hat p = q$)")
ax.axhline(0.5, color="#9a9a9a", lw=0.8, ls=":")
ax.axhline(
    1 / (1 + w),
    color="k",
    lw=0.9,
    ls="--",
    label=rf"$\theta = 1/(1+w)$ = {1 / (1 + w):.3f}",
)
ax.axvline(0.5, color="k", lw=0.6, ls=":")
ax.set(xlabel="true posterior q = P(on | x)", ylabel=r"BCE optimum $\hat p$")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
plt.show()
print(
    f"A bus with q = 0.5 is predicted at p_hat = {0.5 / (0.5 + w * 0.5):.3f}; thresholding at theta = {1 / (1 + w):.3f} restores the q = 0.5 boundary."
)
print(
    f"With the dataset ratio seen in section 2, the trained model's threshold will be about "
    f"{1 / (1 + bus.BUS_STATUS.mean() / (1 - bus.BUS_STATUS.mean())):.3f}."
)

## 3.4 The conflict rule: the label wins

Occasionally $y^{\text{reach}} \ne y^{\text{target}}$, for example when the BFS
does not recover a path the solver did (different source assumptions, a
mis-flagged branch). In the paper's original formulation $w_{\text{off}}$ sat on
whichever term said "0", so every conflict was resolved toward de-energised. The
current loss instead sets $\omega_i \leftarrow 0$ and $\nu_i \leftarrow w_{\text{off}}$
on conflict buses, so the stationary point is $y_i^{\text{target}}$ whether it
is 0 or 1.

Take a healthy-but-islanded bus of our scenario, flip its label to 1 (solver
says energised, BFS says islanded) and look at the gradient on its logit: it is now **negative** (pushing toward energised)
even though the topology term says otherwise. Its voltage, however, is still
driven to zero, because the voltage masks follow the BFS label.

In [ ]:
CB = ROLE["islanded"] if ROLE["islanded"] is not None else ROLE["hit"]
conf = scenario_table(bus, branch, SID, conflict_bus=CB)
display(
    pd.concat(
        [
            tbl.loc[[CB]].assign(case="original label"),
            conf.loc[[CB]].assign(case="label flipped to 1"),
        ]
    ).set_index("case")
)
p_star = stationary_point(
    1.0, 0.5, omega=0.0, nu=conf.attrs["w_off"], y_reach=0, y_target=1
)
print(
    f"Stationary point of bus {conf.bus[CB]}'s logit under the conflict rule: p* = {p_star:.3f} (the label)"
)
print(
    "Conflict-rate metrics reported by the loss:",
    {
        k: round(v, 3)
        for k, v in conf.attrs["terms"].items()
        if "Conflict" in k or "Disagreement" in k
    },
)

## 3.5 The voltage terms

$\mathcal{L}_{\text{off}}$ pulls $\hat V_i$ to zero on de-energised buses with a
quadratic part and a constant pull $\lambda_1$ that closes the last few
millivolts (eq. 24). $\mathcal{L}_{\text{on}}$ pulls $\hat V_i$ to the solver
value on energised buses, weighted by the model's own confidence
$\mathrm{sg}(\hat p_i)$ (eq. 25): weak while the classifier is uncertain,
strong once it is confident, a built-in curriculum. Neither term reaches
$z_i$; the status head is trained only by the two BCE terms.

Look at the `dL/dV_hat` column in section 3.2: large and positive on the
de-energised buses (voltage down), small and positive on energised buses whose
solver voltage is below 1 p.u. (the feeder sags along its length), and zero on
the source bus, which is held at 1 p.u.

# 4. Tuning the loss

Most of the behaviour is fixed by the formulation. The parameters that encode a
real trade-off are the four $\lambda$ weights, $\lambda_1$, and the class-weight
settings. The table below is what each one does to the gradient; the full
reference is `docs/applications/VLD_loss_parameters.md`.

| parameter | moves | effect |
|---|---|---|
| `topology_weight` $\lambda_{\text{reach}}$ | $a_i$ | reference scale; keep at 1 |
| `target_anchor_weight` $\lambda_{\text{anc}}$ | $b_i$ | speed of learning on agreeing buses, and the **only** force on conflict buses |
| `off_vm_weight` $\lambda_{\text{off}}$ | $\partial\mathcal{L}/\partial\hat V$ on off buses | strength of the voltage collapse |
| `on_vm_weight` $\lambda_{\text{on}}$ | $\partial\mathcal{L}/\partial\hat V$ on on buses | voltage fidelity; weighted by confidence |
| `unreachable_l1_weight` $\lambda_1$ | constant pull near $\hat V=0$ | lower if off-bus voltages oscillate around zero |
| `unreachable_pos_weight`, `max_pos_weight`, `pos_weight_momentum` | $w_{\text{off}}$ and hence $\theta$ | class balance; momentum keeps $\theta$ stable across batches |

Let us vary two of them on our `case33bw` scenario and watch the gradients.

In [ ]:
ON, OFF = ROLE["on"], ROLE["hit"] if ROLE["hit"] is not None else ROLE["islanded"]


def name(r):
    """Bus id of a table row position."""
    return int(tbl.bus[r])


rows = []
for lam_anc in (0.25, 0.5, 1.0):
    t = scenario_table(bus, branch, SID, conflict_bus=CB, target_anchor_weight=lam_anc)
    rows.append(
        {
            "target_anchor_weight": lam_anc,
            f"dL/dz bus {name(ON)} (agreeing, on)": t.loc[ON, "dL/dz (status logit)"],
            f"dL/dz bus {name(OFF)} (agreeing, off)": t.loc[
                OFF, "dL/dz (status logit)"
            ],
            f"dL/dz bus {name(CB)} (conflict, label 1)": t.loc[
                CB, "dL/dz (status logit)"
            ],
        }
    )
display(pd.DataFrame(rows).set_index("target_anchor_weight"))
rows = []
for lam_on in (0.25, 0.5, 1.0):
    t = scenario_table(bus, branch, SID, on_vm_weight=lam_on)
    rows.append(
        {
            "on_vm_weight": lam_on,
            f"dL/dV bus {name(ON)} (on, target {t.loc[ON, 'V_true']:.3f})": t.loc[
                ON, "dL/dV_hat"
            ],
            f"dL/dV bus {name(OFF)} (off)": t.loc[OFF, "dL/dV_hat"],
        }
    )
display(pd.DataFrame(rows).set_index("on_vm_weight"))

Raising `target_anchor_weight` speeds up every bus a little and the conflict bus
a lot, because the anchor is all it has. Raising `on_vm_weight` changes only the
energised-bus voltage gradient; the de-energised buses are untouched. That
separation is what makes the loss tunable one knob at a time.

How to read a training run when tuning: `VLD On Recall` low while
`VLD Off Recall` is high means a bias toward de-energisation (check
`VLD Off Class Weight` against the clamp and `VLD Decision Threshold`);
`VLD Conflict Rate (reach 0, label 1)` persistently high means the BFS misses
paths the solver finds (check `reachability_sources`); `VLD On Vm Loss` lagging
means the voltage head is being over-committed to the outage case (lower
`off_vm_weight` or raise `on_vm_weight`). Select checkpoints on
`VLD Balanced Accuracy`, never on off-class F1 alone.

# 5. Train a dummy model

A deliberately tiny `GNS_heterogeneous` (hidden size 32, 2 layers, 2 heads,
6 epochs) trained with the VLD loss alone, through the same `gridfm_app_dist
train` command you would use on a cluster. Expect a few minutes on a T4. The
point is to see the loss components and the derived threshold on real curves,
not to reach paper-level accuracy.

In [ ]:
cfg = read_yaml(TUTORIAL_DIR / "configs" / "tutorial_train.yaml")
cfg["data"]["networks"] = [NETWORK]
cfg["data"]["scenarios"] = [N_SCEN]
cfg["training"]["accelerator"] = DEVICE
TRAIN_CFG = write_yaml(WORK / "tutorial_train.yaml", cfg)
print(yaml.safe_dump(cfg["training"], sort_keys=False))
run(
    python_module(
        "gridfm_app_dist",
        "train",
        "--config",
        TRAIN_CFG,
        "--data_path",
        DATA_DIR,
        "--log_dir",
        LOG_DIR,
        "--exp_name",
        "vld_tutorial_dummy",
        "--run_name",
        "dummy",
        "--num_workers",
        "0",
    ),
    cwd=WORK,
)
DUMMY = latest_run_artifacts(LOG_DIR, "vld_tutorial_dummy")
print("artifacts:", DUMMY)

## 5.1 Read the curves

Each loss component is logged separately on validation. A falling total can
hide a weak term, so look at them one by one, and at the two quantities that
make the threshold: the class weight and the threshold itself.

In [ ]:
plot_history(
    LOG_DIR,
    "vld_tutorial_dummy",
    [
        "Validation VLD Topology BCE",
        "Validation VLD Target Anchor BCE",
        "Validation VLD Off Vm Loss",
        "Validation VLD On Vm Loss",
    ],
    "Validation: the four VLD terms",
)
plt.show()
plot_history(
    LOG_DIR,
    "vld_tutorial_dummy",
    [
        "Validation VLD Balanced Accuracy",
        "Validation VLD Off Recall",
        "Validation VLD On Recall",
        "Validation VLD Off Class Weight",
        "Validation VLD Decision Threshold",
        "Validation VLD Topo-vs-Target Disagreement",
    ],
    "Validation: classification, class weight and derived threshold",
    ncols=3,
)
plt.show()

The training command finishes with a test pass, and because this is the VLD
task the evaluator ran automatically on the test predictions. Its report is in
the run's artifacts:

In [ ]:
eval_dir = next((DUMMY / "test").glob("*_eval"))
display(Markdown((eval_dir / "summary.md").read_text()))
display(Image(filename=str(eval_dir / "confusion_matrix.png"), width=360))
display(Image(filename=str(eval_dir / "probability_separation.png"), width=520))

# 6. Inference with a pre-trained model

The tutorial folder ships a trained checkpoint in `pretrained/` (its training
`config.yaml`, the `best_model_state_dict.pt` and the fitted
`normalizer_stats.pt`). Its config names the dataset it was trained on, so we
re-key the config and the normaliser to the dataset we generated; the weights
are used as they are. Then we run `gridfm_app_dist predict`. Evaluation runs
automatically after the parquet is written.

In [ ]:
PRETRAINED = TUTORIAL_DIR / "pretrained"
for f in ("best_model_state_dict.pt", "normalizer_stats.pt"):
    fp = PRETRAINED / f
    assert fp.exists(), f"{f} missing from {PRETRAINED}; see pretrained/README.md"
    assert not fp.read_bytes()[:40].startswith(b"version https://git-lfs"), (
        f"{f} is a Git LFS pointer, not the weights: run `git lfs pull` in {REPO_DIR}"
    )
P_CFG, P_STATS, P_WEIGHTS = adapt_pretrained(
    PRETRAINED, NETWORK, N_SCEN, WORK / "pretrained_adapted", accelerator=DEVICE
)
run(
    python_module(
        "gridfm_app_dist",
        "predict",
        "--config",
        P_CFG,
        "--model_path",
        P_WEIGHTS,
        "--normalizer_stats",
        P_STATS,
        "--data_path",
        DATA_DIR,
        "--output_path",
        PRED_DIR,
        "--log_dir",
        LOG_DIR,
        "--exp_name",
        "vld_tutorial_pretrained",
        "--num_workers",
        "0",
    ),
    cwd=WORK,
)
pred = pd.read_parquet(PRED_DIR / f"{NETWORK}_predictions.parquet")
THETA = float(pred["threshold"].iloc[0])
print(
    f"{len(pred):,} bus predictions over {pred.scenario.nunique()} scenarios; model threshold theta = {THETA:.4f}"
)
display(pred.head())

## 6.1 The evaluation report

The evaluator scores the model at its own threshold. The metric names mirror the
ones the loss logs, so you can read them against the training curves above. The
`status_pred_threshold_mismatch` line should be 0: it checks that the exported
hard predictions really were made with the derived threshold.

In [ ]:
eval_dir = PRED_DIR / f"{NETWORK}_eval"
display(Markdown((eval_dir / "summary.md").read_text()))
for name, width in [
    ("confusion_matrix", 360),
    ("roc_pr", 640),
    ("probability_separation", 520),
    ("reliability", 360),
    ("scenario_errors", 600),
    ("per_bus", 700),
    ("voltage_status", 400),
]:
    display(Image(filename=str(eval_dir / f"{name}.png"), width=width))

## 6.2 One scenario

Pick a scenario and look at the two heads together. Misclassified buses are
shaded. The three failure modes to look for: a bus called energised with no
path to the source, a de-energised bus whose voltage did not collapse, and an
energised bus with the right status but the wrong voltage.

In [ ]:
errors = (
    pred.assign(
        err=(pred.status_prob >= THETA).astype(int) != pred.status_true.astype(int)
    )
    .groupby("scenario")
    .err.sum()
)
worst = int(errors.idxmax())
clean = (
    int(errors[errors == 0].index[0]) if (errors == 0).any() else int(errors.idxmin())
)
for sid in (clean, worst):
    plot_scenario(pred, sid, THETA)
    plt.show()

# 7. Where to go from here

* **Tune on your own network.** Replace `case33bw` in `configs/tutorial_datagen.yaml`
  with another pandapower factory (`create_cigre_network_lv`, a Kerber network)
  or an Excel/YAML network definition shipped in `contingency_data_gen/configs`,
  regenerate, retrain. The `examples/config/scenario_33radial.yaml` config shows
  the joint loss (`MaskedMSE + PBE + VLDTopologyLoss`) used for the paper.
* **Sweep the two knobs that matter** (`target_anchor_weight` ∈ {0.25, 0.5, 1},
  `on_vm_weight` ∈ {0.25, 0.5}) and select on validation `VLD Balanced Accuracy`.
* **Read the report, not just the accuracy.** `on_recall` is one minus the
  false-de-energisation rate; `ap_deenergised` is the number that moves when
  the detector improves; the reliability curve tells you whether the derived
  threshold's assumption (calibration in the shifted sense) holds.

Equivalent terminal commands for everything the notebook did:

```bash
contingency_data_gen --config examples/tutorial_vld/configs/tutorial_datagen.yaml
gridfm_app_dist train   --config examples/tutorial_vld/configs/tutorial_train.yaml --data_path data --log_dir mlruns
gridfm_app_dist predict --config <adapted config> --model_path <weights> --normalizer_stats <stats> \
                        --data_path data --output_path predictions
python -m gridfm_app_dist.io.vld_evaluation predictions/tutorial_case33bw_predictions.parquet --outdir vld_eval
```

**References.** Kisuule, Bessa, Hernando-Gil, *Topology-Aware Graph Transformers
for detecting voltage loss after distribution network failures* (the loss and
equation numbers used above); `docs/applications/VLD_loss_parameters.md`;
[pandapower networks](https://pandapower.readthedocs.io/en/v2.0.0/networks.html).

In [ ]:
# Keep your results before the Colab session ends.
if IN_COLAB:
    import shutil
    from google.colab import files

    keep = WORK / "results_to_download"
    shutil.rmtree(keep, ignore_errors=True)
    keep.mkdir()
    shutil.copytree(PRED_DIR, keep / "predictions")
    shutil.copytree(DUMMY, keep / "dummy_model_run")
    archive = shutil.make_archive(
        str(WORK.parent / "vld_tutorial_results"), "zip", root_dir=keep
    )
    files.download(archive)